In [ ]:
import numpy as np
from collections import Counter

class ThuatToanKNN:
    """
    Lớp triển khai thuật toán K-Láng giềng gần nhất (K-NN).
    Hỗ trợ 2 cách đánh giá trọng số phiếu bầu của láng giềng: 'dong_deu' và 'khoang_cach'.
    """
    def __init__(self, so_lang_gieng=3, kieu_trong_so='dong_deu'):
        self.k = so_lang_gieng
        self.kieu_trong_so = kieu_trong_so
        self.du_lieu_huan_luyen = None
        self.nhan_huan_luyen = None

    def huan_luyen(self, du_lieu, nhan):
        """
        K-NN là thuật toán Học lười (Lazy Learning).
        Quá trình huấn luyện chỉ đơn giản là lưu trữ lại dữ liệu để so sánh sau này.
        """
        self.du_lieu_huan_luyen = du_lieu
        self.nhan_huan_luyen = nhan

    def tinh_khoang_cach_euclid(self, diem_A, diem_B):
        """
        Tính khoảng cách đường thẳng (Euclid) giữa 2 điểm dữ liệu.
        Được tách riêng từng bước toán học để dễ đọc, tránh code gộp khó hiểu.
        """
        hieu_toa_do = diem_A - diem_B
        binh_phuong_hieu = hieu_toa_do ** 2
        tong_binh_phuong = np.sum(binh_phuong_hieu)
        return np.sqrt(tong_binh_phuong)

    def bau_chon_dong_deu(self, nhan_cac_lang_gieng):
        """
        Cách 1: Mỗi láng giềng có 1 phiếu bầu ngang nhau.
        Trả về nhãn có số lượng nhiều nhất.
        """
        bo_dem = Counter(nhan_cac_lang_gieng)
        nhan_pho_bien_nhat = bo_dem.most_common(1)[0][0]
        return nhan_pho_bien_nhat

    def bau_chon_theo_khoang_cach(self, khoang_cach_lang_gieng, nhan_cac_lang_gieng):
        """
        Cách 2: Láng giềng gần hơn sẽ có trọng số phiếu bầu cao hơn.
        Trọng số = 1 / khoảng cách.
        """
        diem_so_cac_nhan = {}

        for khoang_cach, nhan in zip(khoang_cach_lang_gieng, nhan_cac_lang_gieng):
            # Thêm số cực nhỏ (1e-5) để tránh lỗi chương trình chia cho 0
            # (Trường hợp điểm dự đoán trùng khớp hoàn toàn với 1 điểm đã có)
            trong_so = 1.0 / (khoang_cach + 1e-5)

            # Cộng dồn điểm số cho nhãn tương ứng
            if nhan in diem_so_cac_nhan:
                diem_so_cac_nhan[nhan] += trong_so
            else:
                diem_so_cac_nhan[nhan] = trong_so

        # Tìm và trả về nhãn có tổng trọng số cao nhất
        nhan_chien_thang = max(diem_so_cac_nhan, key=diem_so_cac_nhan.get)
        return nhan_chien_thang

    def du_doan_mot_diem(self, diem_kiem_tra):
        """
        Dự đoán nhãn cho một điểm dữ liệu.
        """
        # Bước 1: Tính khoảng cách từ điểm này đến TOÀN BỘ điểm trong tập huấn luyện
        danh_sach_khoang_cach = []
        for diem_huan_luyen in self.du_lieu_huan_luyen:
            khoang_cach = self.tinh_khoang_cach_euclid(diem_kiem_tra, diem_huan_luyen)
            danh_sach_khoang_cach.append(khoang_cach)

        danh_sach_khoang_cach = np.array(danh_sach_khoang_cach)

        # Bước 2: Tìm chỉ số của k láng giềng có khoảng cách nhỏ nhất
        chi_so_k_diem_gan_nhat = np.argsort(danh_sach_khoang_cach)[:self.k]

        # Bước 3: Lấy ra khoảng cách và nhãn của k láng giềng đó
        khoang_cach_k_diem = danh_sach_khoang_cach[chi_so_k_diem_gan_nhat]
        nhan_k_diem = self.nhan_huan_luyen[chi_so_k_diem_gan_nhat]

        # Bước 4: Tính toán kết quả dựa trên loại trọng số đã chọn
        if self.kieu_trong_so == 'dong_deu':
            return self.bau_chon_dong_deu(nhan_k_diem)
        elif self.kieu_trong_so == 'khoang_cach':
            return self.bau_chon_theo_khoang_cach(khoang_cach_k_diem, nhan_k_diem)

    def du_doan(self, tap_du_lieu_kiem_tra):
        """
        Dự đoán nhãn cho một danh sách (mảng) nhiều điểm dữ liệu.
        """
        nhan_du_doan = []
        for diem in tap_du_lieu_kiem_tra:
            nhan = self.du_doan_mot_diem(diem)
            nhan_du_doan.append(nhan)
        return np.array(nhan_du_doan)


# --- KỊCH BẢN KIỂM TRA CHO PHÉP TỰ NHẬP K VÀ TRỌNG SỐ ---
if __name__ == "__main__":
    # Tập dữ liệu huấn luyện cố định gồm 3 điểm mẫu
    tap_huan_luyen = np.array([
        [2.0, 1.9],  # Rất GẦN điểm cần kiểm tra (Nhãn: Đỏ)
        [3.0, 2.0],  # Hơi xa (Nhãn: Xanh)
        [3.0, 2.1]   # Hơi xa (Nhãn: Xanh)
    ])
    nhan_huan_luyen = np.array(['Đỏ', 'Xanh', 'Xanh'])

    # Điểm dữ liệu mới cần dự đoán nhãn: [2.0, 2.0]
    tap_kiem_tra = np.array([[2.0, 2.0]])

    print("=== CHƯƠNG TRÌNH DỰ ĐOÁN K-NN CHỦ ĐỘNG ===")
    print(f"Số lượng điểm dữ liệu hiện có trong tập huấn luyện: {len(tap_huan_luyen)}")

    # Vòng lặp nhận dữ liệu đầu vào từ bàn phím và kiểm tra tính hợp lệ
    while True:
        try:
            nhap_k = int(input(f"Nhập số lượng láng giềng k (Từ 1 đến {len(tap_huan_luyen)}): "))
            if 1 <= nhap_k <= len(tap_huan_luyen):
                break
            else:
                print(f"Vui lòng nhập số k nằm trong khoảng hợp lệ!")
        except ValueError:
            print("Lỗi: Số láng giềng k phải là một số nguyên!")

    while True:
        print("\nChọn kiểu tính trọng số phiếu bầu:")
        print("1. Loại 'dong_deu' (Đồng đều - Đa số thắng)")
        print("2. Loại 'khoang_cach' (Theo khoảng cách - Gần hơn ăn điểm cao hơn)")
        lua_chon = input("Nhập lựa chọn của bạn (1 hoặc 2): ").strip()

        if lua_chon == '1':
            nhap_trong_so = 'dong_deu'
            break
        elif lua_chon == '2':
            nhap_trong_so = 'khoang_cach'
            break
        else:
            print("Lựa chọn không hợp lệ. Vui lòng nhập lại!")

    # Khởi tạo mô hình dựa trên các thông số người dùng vừa tự nhập
    knn_mo_hinh = ThuatToanKNN(so_lang_gieng=nhap_k, kieu_trong_so=nhap_trong_so)
    knn_mo_hinh.huan_luyen(tap_huan_luyen, nhan_huan_luyen)

    # Thực hiện dự đoán nhãn
    ket_qua = knn_mo_hinh.du_doan(tap_kiem_tra)

    print("\n--- KẾT QUẢ XỬ LÝ ---")
    print(f"Cấu hình chạy: k = {nhap_k}, Trọng số = '{nhap_trong_so}'")
    print(f"Điểm cần dự đoán: {tap_kiem_tra[0]}")
    print(f"==> Nhãn dự đoán cuối cùng là: **{ket_qua[0]}**")